In [1]:
# --- Install khusus untuk BERT ---
!pip install -q transformers datasets scikit-learn pandas

In [ ]:
import os
import IPython

!pip install --upgrade transformers datasets

# Paksa restart runtime otomatis
# os.kill(os.getpid(), 9)

In [ ]:
os.kill(os.getpid(), 9)

In [4]:
# !pip install transformers[torch]
!pip install --upgrade transformers datasets

In [5]:
!pip install torch

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 MB 4.3 MB/s eta 0:00:00m eta 0:00:010:00:01
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 4.4 MB/s eta 0:00:004.7 MB/s eta 0:00:01
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)


In [26]:
# !pip install --upgrade pandas numpy --quiet
!pip install numpy==1.26.4 --quiet


In [29]:
!pip uninstall pandas numpy --quiet
!pip install pandas numpy --quiet

Proceed (Y/n)? ^C
ERROR: Operation cancelled by user


In [5]:
import os
import IPython

from transformers import TrainingArguments
print(TrainingArguments.__module__)

import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch
from datasets import Dataset

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
import numpy as np

transformers.training_args


In [3]:
df = pd.read_csv('4_preprocessed_user_comments.csv')
df.head()

,comment,replies,date,user_name,source_file,link,manual_label,relevansi,clean_text
0,Prediksi Menteri Keuangan Sri Mulyani Indrawat...,"['Tidak butuh prediksi. Tapi solusi', 'Tdk usa...",2025-04-04T07:16:51Z,@kontantv,BERITA_KONTAN_user_comments.csv,https://www.youtube.com/watch?v=p0NTgOcwj3k,netral,relevan,prediksi menteri uang sri mulyani indrawati se...
1,korupsi dan pajak yg tinggi lebih berbahaya dr...,[],2025-04-29T10:38:49Z,@victorferry5496,BERITA_KONTAN_user_comments.csv,https://www.youtube.com/watch?v=p0NTgOcwj3k,negatif,relevan,korupsi pajak bahaya dr tarrif trumph korupsi ...
2,Korups yg di indonesia yg.bikom.parah makanya ...,[],2025-04-29T02:33:56Z,@zannariabintang4812,BERITA_KONTAN_user_comments.csv,https://www.youtube.com/watch?v=p0NTgOcwj3k,negatif,relevan,korups indonesia ygbikomparah prabowo hukum ko...
3,Maap buk di putus aja hububgan jikala tk teken...,[],2025-04-28T13:11:38Z,@fahmiidris4355,BERITA_KONTAN_user_comments.csv,https://www.youtube.com/watch?v=p0NTgOcwj3k,negatif,relevan,maap buk putus hububgan jikala tk teken gitu k...
4,Waduh🫠,[],2025-04-27T02:29:54Z,@PolitikaHistoria,BERITA_KONTAN_user_comments.csv,https://www.youtube.com/watch?v=p0NTgOcwj3k,negatif,relevan,NaN


In [24]:
# !pip install 'transformers[torch]' --quiet
!pip install transformers\[torch\] --quiet

In [11]:
!pip install 'accelerate>=0.26.0'

In [13]:
!pip list | grep accelerate

accelerate                1.7.0


In [14]:
!which python
!which pip

/opt/anaconda3/envs/pi_sentiment_analysis/bin/python
/opt/anaconda3/envs/pi_sentiment_analysis/bin/pip


In [15]:
!pip show accelerate

Name: accelerate
Version: 1.7.0
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The HuggingFace team
Author-email: zach.mueller@huggingface.co
License: Apache
Location: /opt/anaconda3/envs/pi_sentiment_analysis/lib/python3.13/site-packages
Requires: huggingface-hub, numpy, packaging, psutil, pyyaml, safetensors, torch
Required-by: 


# indoBERT

In [6]:
# --- Encode Labels ---
label_encoder = LabelEncoder()
df['labels'] = label_encoder.fit_transform(df['manual_label'])

# --- Tokenizer ---
tokenizer = BertTokenizer.from_pretrained('indobenchmark/indobert-base-p1')

# --- Tokenization Function ---
def tokenize(batch):
    batch['text'] = [str(t) for t in batch['text']]
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

# --- Stratified KFold ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Prepare to collect metrics
results = []

X = df['clean_text'].values
y = df['labels'].values

fold = 1
for train_idx, val_idx in skf.split(X, y):
    print(f"\n=== Fold {fold} ===")

    # Split data
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Convert to Dataset
    train_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_train, 'labels': y_train}))
    val_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_val, 'labels': y_val}))

    # Tokenization
    train_dataset = train_dataset.map(tokenize, batched=True)
    val_dataset = val_dataset.map(tokenize, batched=True)
    train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

    # Model
    model = BertForSequenceClassification.from_pretrained('indobenchmark/indobert-base-p1', num_labels=3)

    # Training Arguments
    training_args = TrainingArguments(
        output_dir=f'./results/fold_{fold}',
        logging_strategy="no",  # Disable logging to make it clean
        num_train_epochs=2,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    # Train
    trainer.train()

    # Predict
    preds_output = trainer.predict(val_dataset)
    y_pred = np.argmax(preds_output.predictions, axis=1)

    # Metrics
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='macro')
    precision = precision_score(y_val, y_pred, average='macro')
    recall = recall_score(y_val, y_pred, average='macro')

    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")

    results.append({
        "fold": fold,
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall
    })

    fold += 1

/opt/anaconda3/envs/pi_sentiment_analysis/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(



=== Fold 1 ===


Map:   0%|          | 0/9172 [00:00<?, ? examples/s]

Map:   0%|          | 0/2293 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,No log,0.119923
2,No log,0.127357


... (22 outputs left out)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,No log,0.100388
2,No log,0.114592


Accuracy: 0.6681 | F1: 0.1602 | Precision: 0.1336 | Recall: 0.2000


/opt/anaconda3/envs/pi_sentiment_analysis/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [7]:
# --- Overall Summary ---
df_results = pd.DataFrame(results)
print("\nAverage Scores:")
print(df_results.mean())


Average Scores:
fold         3.000000
accuracy     0.668469
f1_score     0.204382
precision    0.176873
recall       0.249254
dtype: float64


In [9]:
df_results.head()

,fold,accuracy,f1_score,precision,recall
0,1,0.668120,0.200261,0.167030,0.250000
1,2,0.668120,0.200261,0.167030,0.250000
2,3,0.669865,0.260917,0.249652,0.296271
3,4,0.668120,0.200261,0.167030,0.250000
4,5,0.668120,0.160209,0.133624,0.200000


In [10]:
df.head()

,comment,replies,date,user_name,source_file,link,manual_label,relevansi,clean_text,labels
0,Prediksi Menteri Keuangan Sri Mulyani Indrawat...,"['Tidak butuh prediksi. Tapi solusi', 'Tdk usa...",2025-04-04T07:16:51Z,@kontantv,BERITA_KONTAN_user_comments.csv,https://www.youtube.com/watch?v=p0NTgOcwj3k,netral,relevan,prediksi menteri uang sri mulyani indrawati se...,2
1,korupsi dan pajak yg tinggi lebih berbahaya dr...,[],2025-04-29T10:38:49Z,@victorferry5496,BERITA_KONTAN_user_comments.csv,https://www.youtube.com/watch?v=p0NTgOcwj3k,negatif,relevan,korupsi pajak bahaya dr tarrif trumph korupsi ...,1
2,Korups yg di indonesia yg.bikom.parah makanya ...,[],2025-04-29T02:33:56Z,@zannariabintang4812,BERITA_KONTAN_user_comments.csv,https://www.youtube.com/watch?v=p0NTgOcwj3k,negatif,relevan,korups indonesia ygbikomparah prabowo hukum ko...,1
3,Maap buk di putus aja hububgan jikala tk teken...,[],2025-04-28T13:11:38Z,@fahmiidris4355,BERITA_KONTAN_user_comments.csv,https://www.youtube.com/watch?v=p0NTgOcwj3k,negatif,relevan,maap buk putus hububgan jikala tk teken gitu k...,1
4,Waduh🫠,[],2025-04-27T02:29:54Z,@PolitikaHistoria,BERITA_KONTAN_user_comments.csv,https://www.youtube.com/watch?v=p0NTgOcwj3k,negatif,relevan,NaN,1
